-------------------
# 01: SQI Analysis & Supervised Learning:

In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib qt
from ma.qa_ma_sqi import calculate_sqis, save_sqi_df, load_sqi_df, SQIS, SupervisedLearning
from ma.utils import load_all_signals
from ma.qa_ma_visu import feature_scatter_plot, feature_violin_plot
import ipywidgets as widgets
import numpy as np

LABEL = "sq" # ma or sq
SIGNAL_TYPE = "MI"
CLASS_NAME = "RR"
SQ_FEATURES_ECG_HR = [feature for feature in SQIS if feature not in ["wavelet_energy", "wavelet_entropy", "wavelet_std", "wavelet_mean"]]
MA_FEATURES_ECG_HR = ["skew", "f80", "zc_mean", "mean_dist_of_peaks", "peak_freq","mean_dist_maf_peaks", "std_dist_maf_peaks"]
SQ_FEATURES_PPG_HR = ["zc_mean", "zc_derivative", "skew", "zc", "cluster_number_maf", "f80", "peak_freq", "kurt"]
MA_FEATURES_PPG_HR = ["skew", "cluster_number_maf","zc_mean", "kurt", "f80", "zc", "iqr", "zc_derivative", "mean_dist_maf_peaks", "std_dist_maf_peaks", "wavelet_entropy", "wavelet_energy"]

SQ_FEATURES_SCG_RR = ["mean_dist_maf_peaks", "wavelet_entropy", "mean_dist_of_peaks", "wavelet_energy", "std_dist_maf_peaks", "rms", "std", "wavelet_std", "peak", "peak_amp","mean_amp","f80"]#, "mavsd","mavfd"]#	cluster_number_maf	med_amp	wavelet_mean	mean	iqr	peak_freq	skew	kurt	zc	zc_derivative	zc_mean
MA_FEATURES_SCG_RR = ["kurt", "zc_mean", "skew", "std_dist_maf_peaks", "mean", "wavelet_mean", "zc_derivative", "zc", "cluster_number_maf", "peak_freq", "f80" ,"peak_amp","rms","wavelet_std","std","mean_amp","iqr","peak","med_amp","mavfd"]#,"mavsd","mean_dist_of_peaks","wavelet_energy","mean_dist_maf_peaks","wavelet_entropy"]
SQ_FEATURES_MI_RR = ["skew","f80","cluster_number_maf","mean_dist_maf_peaks","std_dist_maf_peaks","zc_mean","zc","kurt","peak_freq","rms","std","mean_amp","peak_amp"]#,"mavfd","mavsd","mean","iqr","peak","med_amp","mean_dist_of_peaks"]
MA_FEATURES_MI_RR = ["skew","f80","zc_mean","mean_dist_of_peaks","peak_freq","mean_dist_maf_peaks","std_dist_maf_peaks","mavfd","mavsd"]#,"peak_amp"]#"mean_amp","rms","std","kurt","peak","med_amp","iqr","zc","mean","cluster_number_maf"]

match CLASS_NAME:
    case "HR":
        match SIGNAL_TYPE:
            case "ECG":
                FEATURES = SQ_FEATURES_ECG_HR if LABEL == "sq" else MA_FEATURES_ECG_HR
            case "PPG":
                FEATURES = SQ_FEATURES_PPG_HR if LABEL == "sq" else MA_FEATURES_PPG_HR
            case "SCG":
                FEATURES = []
            case "MI":
                FEATURES = []
    case "RR":
        match SIGNAL_TYPE:
            case "ECG":
                FEATURES = []
            case "PPG":
                FEATURES = []
            case "SCG":
                FEATURES = SQ_FEATURES_SCG_RR if LABEL == "sq" else MA_FEATURES_SCG_RR
            case "MI":
                FEATURES = SQ_FEATURES_MI_RR if LABEL == "sq" else MA_FEATURES_MI_RR


In [ ]:
# Calculate SQIs:
# Get the discriminated data:
# all_data = load_all_signals(signal_type=SIGNAL_TYPE, bandpassed=True, normalized=True, class_name=CLASS_NAME)
# sqi_df = calculate_sqis(data=all_data)
# save_sqi_df(sqi_df, signal=SIGNAL_TYPE, class_name=CLASS_NAME)

sqi_df = load_sqi_df(signal=SIGNAL_TYPE,label=LABEL, class_name=CLASS_NAME)
sqi_df

In [ ]:
interactive_feature_scatter_plot = widgets.interactive(feature_scatter_plot, feature1 = SQIS, feature2 = SQIS, sqi_df = widgets.fixed(sqi_df), label = widgets.fixed(LABEL), save_image = False)
interactive_feature_scatter_plot

In [ ]:
interactive_feature_violin_plot = widgets.interactive(feature_violin_plot, feature1 = SQIS, sqi_df = widgets.fixed(sqi_df), label = widgets.fixed(LABEL), save_image = False)
interactive_feature_violin_plot

### 2.2 Supervised Learning:
Passable Parameters:
- sqi_df: pd.DataFrame. The dataframe that holds the sqi values.
- standardize: bool. If true, the features will be standardized before train.
- test_parti: list. The defined participants will be seperated to test dataset. If not defined, it will randomly select 30% of the data to seperation.

In [ ]:
sL = SupervisedLearning(sqi_df=sqi_df, label=LABEL, features = FEATURES, class_name=CLASS_NAME)

In [ ]:
sL.evaluate_models(cross_validation=False)

In [ ]:
sL.evaluate_models(cross_validation=True)

In [ ]:
sL.predict(x=sL._X_train, ground_truth=sL._y_train)

In [ ]:
sL.predict(x=sL._X_test, ground_truth=sL._y_test)

In [24]:
sL.save_model(model_name="rusboosted_random_forests", file_name=f"{SIGNAL_TYPE}_{LABEL}_{CLASS_NAME}_rusboosted_random_forests")

-------------------
# 02: Multi-Layer Perceptron (MLP) Neural Networks

In [ ]:
%load_ext autoreload
%autoreload 2
from ma.qa_ma_utils import load_train_test_datasets
from ma.qa_ma_networks import MLP
SIGNAL_TYPE = "SCG"
LABEL = "sq" # ma or sq
CLASS_NAME = "RR"

## Step 1: Data Preperation
- Passable Parameters:
    - `signal_type`: ECG - PPG - SCG - MI 
        - Gathers all 4 (12 for SCG) signals together. For example: [P1:ECG1:1 , P1:ECG2:1, P1:ECG3: 1, P1:ECG4: 1 .... P20:ECG4: last signal]
    - `class_name`: HR - RR
    - `validation_parti`: Optional. If not given, random participants will be seperated to use in validation dataset.


In [ ]:
datasets = load_train_test_datasets(SIGNAL_TYPE, label=LABEL, class_name=CLASS_NAME, shuffle_train_data=True)
# datasets["X_train"][0]

## Step 2: Define the MLP Model:
- Passable Parameters:
    - `datasets`: MUST be in the dictionary format and include X_train, X_test, y_train, y_test.
    - `depth`: Number of hidden layers.
    - `activations`: List of activation functions. If the number of provided activition functions does not match with the depth, it uses the first activation function for all hidden layers! Example: tanh - relu - selu ...

For further information & configuration, please check out the class itself!

In [ ]:
mlp = MLP(datasets = datasets, label=LABEL, depth = 8, activations = ["relu"]*8, learning_rate = 0.001)
mlp._network.summary()

## Step 3: Train the Model:
- Passable Parameters:
    - `epochs`: Optional. Number of complete pass through the entire dataset.
    - `batch_size`: Optional. Number of data in each batch.
    - `validation_split`: Optional. Split percentage of train & validation set.
    - `callback`: Optional. If true, after each epoch a self-configured log would be displayed.

In [ ]:
mlp.train_network(epochs=100, thresholds={"ma_acc": 0.7, "no_ma_acc": 0.7})

## Step 4: Test Data:

#### 4.2: Predict & Evaluate Any Other Data (here used the same data only for showing purposes):

In [ ]:
# Prediction:
results = mlp.predict(X = mlp.X_test, ground_truth= mlp.y_test)
results

In [8]:
mlp.save_network(name= f"{SIGNAL_TYPE}_{LABEL}_{CLASS_NAME}_mlp")

# 03: Deep Learning:

### 3.1: CNNs
Passable Parameters:
- datasets: Return of the `ma.qa_utils.load_train_test_datasets` function.
- depth: Number of Convolutional Blocks. [ Conv -> Conv -> Pool].
- filters: List of number of filters in each Conv block. Length of the filters must be equal to the depth!
    - It describes the amount of different kernels to be used in the convolutional operations. Therefore it also describes the output depth.
- activation: Activation function. More detailed specifications should be done in the class codes.
- padding : Padding size in the convolutional layers.
- kernel_size: Size of the convolutional kernels.
- pool_size: Size of the pooling layers.
- stride: Stride of the pooling layers.

In [ ]:
%load_ext autoreload
%autoreload 2
from ma.qa_ma_utils import load_train_test_datasets
from ma.qa_ma_networks import CNN
LABEL = "sq" # ma or sq
CLASS_NAME = "RR"
datasets = load_train_test_datasets("SCG", label = LABEL, class_name=CLASS_NAME, shuffle_train_data=True)#, test_parti=[1,20])

In [ ]:
# Initialize the network:
cnn = CNN(datasets=datasets, label = LABEL, depth = 4, filters = [8,16,32,32])

In [ ]:
# Train the network
history = cnn.train_network(epochs = 50, batch_size = 32)
history.history.keys(), history.history.values()

In [ ]:
# Evaluate the network
eval = cnn.evaluate_network()

In [ ]:
# Predict Data:
predictions = cnn.predict(X = cnn.X_test, ground_truth= cnn.y_test)
predictions

In [ ]:
# Save the Network:
cnn.save_network(name = "cnn_test")

In [ ]:
# Load the network:
from tensorflow.keras.models import model_from_json
with open('cnn_test.json', 'r') as json_file:
    loaded_model_json = json_file.read()
loaded_model = model_from_json(loaded_model_json)
loaded_model.summary()

### 3.2 : RNNs

In [ ]:
%load_ext autoreload
%autoreload 2
from ma.qa_ma_utils import load_train_test_datasets
from ma.qa_ma_networks import RNN
LABEL = "ma" # ma or sq
CLASS_NAME = "RR"
datasets = load_train_test_datasets("ECG", label = LABEL, class_name=CLASS_NAME, shuffle_train_data=False)

In [ ]:
# Initialize the network:
rnn = RNN(datasets=datasets, label = LABEL, depth = 3)
rnn._network.summary()

In [ ]:
# Train the network
rnn.train_network(epochs = 1, batch_size = 32)

In [ ]:
# Evaluate the network
eval = rnn.evaluate_network()

In [ ]:
# Predict Data:
predictions = rnn.predict(X = rnn.X_test, ground_truth= rnn.y_test)
predictions

In [ ]:
# Save the Network:
rnn.save_network(name = "rnn_test")

In [ ]:
# Load the network:
from tensorflow.keras.models import model_from_json
with open('rnn_test.json', 'r') as json_file:
    loaded_model_json = json_file.read()
loaded_model = model_from_json(loaded_model_json)
loaded_model.summary()

### 3.2 : LSTMs

In [ ]:
%load_ext autoreload
%autoreload 2
from ma.qa_ma_utils import load_train_test_datasets
from ma.qa_ma_networks import LongShortTermMemory
LABEL = "ma" # ma or sq
CLASS_NAME = "RR"
datasets = load_train_test_datasets("ECG", label = LABEL, class_name=CLASS_NAME, shuffle_train_data=False)

In [ ]:
# Initialize the network:
lstm = LongShortTermMemory(datasets=datasets, label = LABEL, depth = 3)
lstm._network.summary()

In [ ]:
# Train the network
lstm.train_network(epochs = 50, batch_size = 32)

In [ ]:
# Evaluate the network
eval = lstm.evaluate_network()

In [ ]:
# Predict Data:
predictions = lstm.predict(X = lstm.X_test, ground_truth= lstm.y_test)
predictions

In [ ]:
# Save the Network:
lstm.save_network(name ="lstm")

In [ ]:
# Load the network:
from tensorflow.keras.models import model_from_json
with open('lstm.json', 'r') as json_file:
    loaded_model_json = json_file.read()
loaded_model = model_from_json(loaded_model_json)
loaded_model.summary()